In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import json
from typing import TYPE_CHECKING, Dict, List, Optional, Sequence, Tuple, Union
from tqdm import tqdm
import pickle

def read_json(data_path):
    try:
        with open(data_path , 'r') as file:
            json_data = file.read()
        data = json.loads(json_data)
        print('try') 
    except:
        data = [json.loads(line) for line in open(data_path)]
        print('except')
    return data


def read_pkl(fname):
    with open(fname, 'rb') as fo:
        pkl_data = pickle.load(fo, encoding='bytes')
    return pkl_data

def write_pkl(pkl_data, fname):
    fo = open(fname, 'wb')
    pickle.dump(pkl_data, fo)
    print("pkl_file write over!")


model_path = './pre_trained_model/llama2-13b-hf'
tokenizer = AutoTokenizer.from_pretrained(model_path)
data_path = './deita_sota_pool/deita_sota_pool_305566.json'
context_limit = 4096
data = read_json(data_path)

try


In [2]:
data[0]

{'id': 'LbHON3Q',
 'source': 'ShareGPT',
 'conversations': [{'from': 'human',
   'value': 'People may now think of queries and replies in novel and creative ways thanks to new AI text systems. What are some crucial inquiries we may pose to these text-based AIs that would yield original and admirable insights into humanity? Make six specific questions that don’t contain the phrase “AI.”'},
  {'from': 'gpt',
   'value': '1. What is the meaning of life and how does it differ from individual to individual?\n2. How does our understanding of the world shape our beliefs and values?\n3. What are the most important qualities that make us human?\n4. How do our emotions influence our decision-making processes and behavior?\n5. What is the relationship between our thoughts and actions, and how does it impact our daily lives?\n6. Can humans find happiness and fulfillment by pursuing material wealth, or is there something more to life?'},
  {'from': 'human', 'value': 'Continue writing please'},
  {'

In [3]:
# example
s = """logger.warning("Loading dataset from disk will ignore other data arguments.")"""
d=tokenizer(s)
len(d['input_ids'])

15

In [5]:
def cut_conversations(conversations: Dict, context_limit: int) -> Dict:
    cur_len = 0
    convs = conversations['conversations']
    updated_conversations = {}
    
    for k,v in conversations.items():
        if k =='conversations':
            updated_conversations['conversations'] = []
        else:
            updated_conversations[k] = v
    
    for conv in convs:        
        tmp_len = len(tokenizer(conv['value'])['input_ids'])
        if conv['from'] == 'human':
            if tmp_len + cur_len > context_limit:
                break
            else:
                cur_len = cur_len + tmp_len
                updated_conversations['conversations'].append( conv )
                

        if conv['from'] == 'gpt':
            if tmp_len + cur_len > context_limit:
                updated_conversations['conversations'] = updated_conversations['conversations'][:-1]
                break
            else:
                cur_len = cur_len + tmp_len
                updated_conversations['conversations'].append( conv )           

    return updated_conversations

In [10]:
exceed_len_list = []
updated_data = []
for index,item in enumerate(tqdm(data)):

    updated_conversations = cut_conversations(item, context_limit)
    if len(updated_conversations['conversations']) < 2 or len(updated_conversations['conversations'])%2 != 0:
        exceed_len_list.append(index)
        # updated_data.append(updated_conversations)
    else:
        updated_data.append(updated_conversations)


100%|██████████| 305566/305566 [13:25<00:00, 379.46it/s]


In [15]:
updated_data[0]

{'id': 'LbHON3Q',
 'source': 'ShareGPT',
 'conversations': [{'from': 'human',
   'value': 'People may now think of queries and replies in novel and creative ways thanks to new AI text systems. What are some crucial inquiries we may pose to these text-based AIs that would yield original and admirable insights into humanity? Make six specific questions that don’t contain the phrase “AI.”'},
  {'from': 'gpt',
   'value': '1. What is the meaning of life and how does it differ from individual to individual?\n2. How does our understanding of the world shape our beliefs and values?\n3. What are the most important qualities that make us human?\n4. How do our emotions influence our decision-making processes and behavior?\n5. What is the relationship between our thoughts and actions, and how does it impact our daily lives?\n6. Can humans find happiness and fulfillment by pursuing material wealth, or is there something more to life?'},
  {'from': 'human', 'value': 'Continue writing please'},
  {'

In [14]:
import json
filename = './data/deita_sota_pool/deita_sota_pool_305263.json'
with open(filename, 'w') as f:
    json.dump(updated_data, f)